# v24 — Factorization Machine PyTorch

Protocolo fixado em 09/11/2026 antes dos treinos; pesquisa termina em 14/11/2026.
Usar somente input/train.csv e input/test.csv. Nenhuma biblioteca nova.
Sem carregar previsões, OOFs, checkpoints ou modelos. Sem SK_ID_CURR como
variável, sem transformações de ID e sem regras manuais nas previsões.

FM de segunda ordem: termo linear e fatores de posto 16, perda BCE sem pesos,
AdamW lr=0,002, weight_decay=0,001, batch=2048, seed=42 e duas threads.
Variáveis plusx da v21, categorias one-hot; numéricas contínuas com quantis
normais (256 quantis), clip [-4,4], ausentes=0 com indicadores. Ajustar tudo
somente no treino correspondente. Inicialização dos fatores N(0,0,01), termo
linear zero e intercepto logit da prevalência do treino.

Para cada parte externa, reservar 15% de seu treino (estratificado, seed 42)
para escolher épocas: até 40, patience 6, min_delta 1e-6 por AUC. Recalcular
features supervisionadas dentro desse treino interno. Recomeçar do zero no
treino externo completo com as épocas escolhidas. Épocas finais = média
arredondada das épocas internas, sem multiplicador. Nenhum rótulo da parte
externa escolhe épocas da FM. Os sete componentes v21 e seu early stopping
original são preservados para reproduzir a referência; seu pré-processamento
não supervisionado herdado usa covariáveis train+test.

Cinco partes StratifiedKFold(5,shuffle=True,random_state=42), ordem original.
Estratos exclusivamente diagnósticos: chave round(crédito/prestação,2),
frequência contada apenas no treino externo; 0–9 (raras, inclui inéditas),
10–49, 50–199, >=200; ausentes/inválidas separados (crédito/prestação não
positivos ou não finitos). Congelar atribuição a estratos antes do primeiro
fit. Não usar estratos para features, pesos de treino ou regras de previsão.
Reportar linhas, %, positivos, AUC v21, FM e ensemble candidato por estrato.

Grade fixa de peso FM [0%,5%,10%,15%,20%], preservando proporções v21;
empates preferem menor peso. Percentis ao quadrado, como v21/v18. Selecionar
somente em 0–1. Corte rápido com percentis das OOF disponíveis: se peso zero
ou sem ganho conjunto 0–1, parar sem treinar 2–4 ou gerar CSV. Se passar,
completar OOF e refixar peso na mesma grade usando somente rótulos 0–1 e
percentis globais; congelar antes de calcular AUCs 2–4/total. Sem retuning.
Reportar AUC FM isolada e Spearman com cada um dos sete componentes.

Promoção exige delta 2–4 >=0,00010 e delta total >0 sobre v21 reproduzida
(2–4=0,7747809092432562; total=0,7740567649584406). Estratos não acrescentam
um critério de promoção; servem para testar a justificativa. Se falhar,
nenhum CSV. Se passar, notebook final treina tudo em 100% do treino;
duas execuções novas, mesmo SHA-256 e menos de 60 min por execução completa
em CPU. Tempos de Colab precisam de verificação; macOS não prova identidade
entre plataformas. Esta pesquisa usa dois workers de CV, duas threads cada;
o treino final será sequencial com duas threads.

Este notebook de pesquisa não escreve CSV. Somente após aprovação será
preparado o notebook final com receita congelada. Os folds já foram usados
em pesquisas anteriores e não são um holdout intocado.

Referência do mecanismo: https://libfm.org/ . Não são usados dados externos.


In [1]:
import os, sys, subprocess, importlib.metadata, random, platform
from pathlib import Path
notebook_wall_started = __import__("time").time()
for key in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
            "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"]:
    os.environ[key] = "2"
random.seed(42)
PINNED = {"numpy": "2.3.5", "pandas": "2.2.3", "scipy": "1.17.0",
          "scikit-learn": "1.8.0", "lightgbm": "4.6.0", "catboost": "1.2.8",
          "torch": "2.6.0", "tabm": "0.0.3", "rtdl-num-embeddings": "0.0.12",
          "threadpoolctl": "3.6.0", "joblib": "1.6.0"}
needed = []
for name, version in PINNED.items():
    try:
        ok = importlib.metadata.version(name).split("+")[0] == version
    except importlib.metadata.PackageNotFoundError:
        ok = False
    if not ok:
        needed.append(f"{name}=={version}")
if needed:
    # O índice CPU é usado para PyTorch em Linux/Colab; macOS usa PyPI.
    torch_needed = [p for p in needed if p.startswith("torch==")]
    other_needed = [p for p in needed if not p.startswith("torch==")]
    if torch_needed:
        command = [sys.executable, "-m", "pip", "install", "--quiet", *torch_needed]
        if sys.platform == "linux":
            command += ["--index-url", "https://download.pytorch.org/whl/cpu"]
        subprocess.run(command, check=True)
    if other_needed:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *other_needed], check=True)
assert {p: importlib.metadata.version(p).split("+")[0] for p in PINNED} == PINNED
# PyTorch fornece libomp no macOS; importar primeiro também resolve LightGBM
# em instalações sem o pacote libomp do Homebrew. Sem alterar as funções v17.
os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".mpl-cache"))
import torch
BASE_DIR = Path(os.environ.get("GCI_WORKDIR", Path.cwd())).resolve()
assert (BASE_DIR / "input/train.csv").is_file(), f"Falta input/train.csv em {BASE_DIR}"
assert (BASE_DIR / "input/test.csv").is_file(), f"Falta input/test.csv em {BASE_DIR}"


## Funções originais da v17
As células seguintes são cópias exatas das definições, sem alterações.

In [2]:
import os, time, json, hashlib, warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
warnings.filterwarnings("ignore")

SEED = 42
N_FOLDS = 5
N_JOBS = 2          # fixo: o resultado não depende do nº de núcleos da máquina
PRIOR = 0.0807      # taxa média de inadimplência no treino (usada na suavização)

In [3]:
def make_features(df):
    df = df.copy()
    df["DAYS_EMPLOYED_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)   # 365243 = código, não valor
    df.loc[df["DAYS_EMPLOYED"] == 365243, "DAYS_EMPLOYED"] = np.nan
    df.loc[df["CODE_GENDER"] == "XNA", "CODE_GENDER"] = np.nan
    df.loc[df["ORGANIZATION_TYPE"] == "XNA", "ORGANIZATION_TYPE"] = np.nan
    df.loc[df["OWN_CAR_AGE"] >= 60, "OWN_CAR_AGE"] = np.nan
    df["CREDIT_ANNUITY_RATIO"] = df["AMT_CREDIT"] / df["AMT_ANNUITY"]       # ≈ prazo (mistura prazo e juros)
    df["ANNUITY_INCOME_RATIO"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]
    df["CREDIT_INCOME_RATIO"] = df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]
    df["CREDIT_GOODS_RATIO"] = df["AMT_CREDIT"] / df["AMT_GOODS_PRICE"]
    df["CREDIT_GOODS_DIFF"] = df["AMT_CREDIT"] - df["AMT_GOODS_PRICE"]
    df["INCOME_PER_PERSON"] = df["AMT_INCOME_TOTAL"] / df["CNT_FAM_MEMBERS"]
    df["AGE_YEARS"] = -df["DAYS_BIRTH"] / 365.25
    df["EMPLOYED_AGE_RATIO"] = df["DAYS_EMPLOYED"] / df["DAYS_BIRTH"]
    df["ID_AGE_RATIO"] = df["DAYS_ID_PUBLISH"] / df["DAYS_BIRTH"]
    df["PHONE_AGE_RATIO"] = df["DAYS_LAST_PHONE_CHANGE"] / df["DAYS_BIRTH"]
    df["CAR_AGE_RATIO"] = df["OWN_CAR_AGE"] / df["AGE_YEARS"]
    return df

In [4]:
def person_groups(raw):
    k = pd.DataFrame({"ID": raw["DAYS_BIRTH"] - raw["DAYS_ID_PUBLISH"],
                      "REG": raw["DAYS_BIRTH"] - raw["DAYS_REGISTRATION"],
                      "PHONE": raw["DAYS_BIRTH"] - raw["DAYS_LAST_PHONE_CHANGE"],
                      "G": raw["CODE_GENDER"]})
    groups = {}
    for name, cols in {"A": ["ID", "REG", "G"], "B": ["ID", "PHONE", "G"]}.items():
        g = k.groupby(cols).ngroup().to_numpy().copy()
        alone = g < 0                                   # faltante → grupo de um pedido só
        g[alone] = g.max() + 1 + np.arange(alone.sum())
        groups[name] = g
    return groups

def person_features(raw, groups):
    f = pd.DataFrame(index=raw.index)
    age_days = -raw["DAYS_BIRTH"].values
    for name, g in groups.items():
        s = pd.Series(g)
        f[f"P{name}_CNT"] = s.map(s.value_counts()).values
        order = pd.DataFrame({"g": g, "d": age_days})
        f[f"P{name}_RANK"] = order.groupby("g")["d"].rank(method="first").values - 1
        srt = order.sort_values(["g", "d"])
        f[f"P{name}_PREV_GAP"] = srt.groupby("g")["d"].diff().reindex(order.index).values
        f[f"P{name}_NEXT_GAP"] = (-srt.groupby("g")["d"].diff(-1)).reindex(order.index).values
        for c in ["AMT_CREDIT", "AMT_INCOME_TOTAL", "EXT_SOURCE_2"]:
            f[f"P{name}_{c}_REL"] = raw[c].values / pd.Series(raw[c].values).groupby(g).transform("mean").values
    return f

In [5]:
def tariff_counts(raw):
    ratio = raw["AMT_CREDIT"] / raw["AMT_ANNUITY"]
    t = pd.DataFrame(index=raw.index)
    for d in [1, 2, 3]:
        k = ratio.round(d)
        t[f"Q{d}_COUNT"] = k.map(k.value_counts()).values
    for c in ["AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE", "AMT_INCOME_TOTAL"]:
        t[c + "_VCOUNT"] = raw[c].map(raw[c].value_counts()).values
    t["Q_FRAC"] = (ratio * 1000).round() % 1000
    return t

def _codes(*cols):
    key = cols[0].astype(str)
    for c in cols[1:]:
        key = key + "|" + c.astype(str)
    return pd.factorize(key)[0]

def tariff_keys(raw):
    ratio = raw["AMT_CREDIT"] / raw["AMT_ANNUITY"]
    return {f"Q{d}": ratio.round(d).fillna(-1).values for d in [1, 2, 3, 4]}

In [6]:
def enhanced_features(raw, base):
    out = base.copy()
    ext_cols = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
    ext = raw[ext_cols]
    for agg in ["mean", "min", "max", "std", "median"]:
        out["EXT_" + agg.upper()] = getattr(ext, agg)(axis=1)
    out["EXT_N_VALID"] = ext.notna().sum(axis=1)
    out["EXT_RANGE"] = ext.max(axis=1) - ext.min(axis=1)
    out["EXT_PRODUCT"] = ext.prod(axis=1, min_count=3)
    for a, b in [(1, 2), (1, 3), (2, 3)]:
        x, z = raw[f"EXT_SOURCE_{a}"], raw[f"EXT_SOURCE_{b}"]
        out[f"EXT_{a}{b}_PRODUCT"] = x * z
        out[f"EXT_{a}{b}_MEAN"] = (x + z) / 2
        out[f"EXT_{a}{b}_DIFF"] = x - z
    out["N_MISSING_RAW"] = raw.isna().sum(axis=1)
    out["EXT_2_AGE"] = raw["EXT_SOURCE_2"] * (-raw["DAYS_BIRTH"] / 365.25)
    out["EXT_3_AGE"] = raw["EXT_SOURCE_3"] * (-raw["DAYS_BIRTH"] / 365.25)
    def ratio(name, num, den):
        out[name] = num / den.replace(0, np.nan)
    ratio("ANNUITY_GOODS_RATIO", raw.AMT_ANNUITY, raw.AMT_GOODS_PRICE)
    ratio("GOODS_INCOME_RATIO", raw.AMT_GOODS_PRICE, raw.AMT_INCOME_TOTAL)
    ratio("GOODS_CREDIT_RATIO", raw.AMT_GOODS_PRICE, raw.AMT_CREDIT)
    ratio("INCOME_ANNUITY_RATIO", raw.AMT_INCOME_TOTAL, raw.AMT_ANNUITY)
    ratio("CREDIT_PER_PERSON", raw.AMT_CREDIT, raw.CNT_FAM_MEMBERS)
    ratio("ANNUITY_PER_PERSON", raw.AMT_ANNUITY, raw.CNT_FAM_MEMBERS)
    out["INCOME_AFTER_ANNUITY"] = raw.AMT_INCOME_TOTAL - raw.AMT_ANNUITY
    ratio("INCOME_AFTER_ANNUITY_PERSON", out.INCOME_AFTER_ANNUITY, raw.CNT_FAM_MEMBERS)
    out["N_ADULTS"] = raw.CNT_FAM_MEMBERS - raw.CNT_CHILDREN
    ratio("CHILDREN_FAMILY_RATIO", raw.CNT_CHILDREN, raw.CNT_FAM_MEMBERS)
    out["AGE_AT_ID"] = raw.DAYS_ID_PUBLISH - raw.DAYS_BIRTH
    out["AGE_AT_REGISTRATION"] = raw.DAYS_REGISTRATION - raw.DAYS_BIRTH
    out["AGE_AT_PHONE"] = raw.DAYS_LAST_PHONE_CHANGE - raw.DAYS_BIRTH
    out["ID_PHONE_GAP"] = raw.DAYS_ID_PUBLISH - raw.DAYS_LAST_PHONE_CHANGE
    out["REGISTRATION_ID_GAP"] = raw.DAYS_REGISTRATION - raw.DAYS_ID_PUBLISH
    out["EMPLOYED_PHONE_GAP"] = out.DAYS_EMPLOYED - raw.DAYS_LAST_PHONE_CHANGE
    ratio("EMPLOYED_PHONE_RATIO", out.DAYS_EMPLOYED, raw.DAYS_LAST_PHONE_CHANGE)
    for c in ext_cols + ["OWN_CAR_AGE", "AMT_REQ_CREDIT_BUREAU_YEAR"]:
        out[c + "_MISSING"] = raw[c].isna().astype("int8")
    out["BUREAU_SUM"] = raw[["AMT_REQ_CREDIT_BUREAU_HOUR", "AMT_REQ_CREDIT_BUREAU_MON", "AMT_REQ_CREDIT_BUREAU_QRT",
                             "AMT_REQ_CREDIT_BUREAU_YEAR"]].sum(axis=1, min_count=1)
    for group_cols in [["NAME_INCOME_TYPE"], ["OCCUPATION_TYPE"], ["ORGANIZATION_TYPE"], ["NAME_EDUCATION_TYPE", "CODE_GENDER"]]:
        key = raw[group_cols].fillna("__NA__").astype(str).agg("|".join, axis=1)
        tag = "_".join(group_cols)
        for c in ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY"]:
            med = raw[c].groupby(key).transform("median")
            out[c + "_GROUP_REL_" + tag] = raw[c] / med.replace(0, np.nan)
    return out.replace([np.inf, -np.inf], np.nan)

In [7]:
def ext_aux_features(X_num, raw, seed=SEED):
    out = pd.DataFrame(index=raw.index)
    params = dict(objective="regression", learning_rate=0.05, num_leaves=31, min_child_samples=100,
                  feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=1, n_estimators=400,
                  deterministic=True, force_row_wise=True, verbose=-1, n_jobs=N_JOBS, random_state=seed)
    ext_cols = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
    drop = [c for c in X_num.columns if c.startswith("EXT_") or "EXT_SOURCE" in c]
    F = X_num.drop(columns=drop)
    for c in ext_cols:
        target = raw[c].values
        has = ~np.isnan(target)
        idx = np.where(has)[0]; miss = np.where(~has)[0]
        pred = np.full(len(raw), np.nan); pred_miss = np.zeros(len(miss))
        others = [o for o in ext_cols if o != c]
        Fc = pd.concat([F, raw[others].reset_index(drop=True)], axis=1)
        for a, b in KFold(5, shuffle=True, random_state=seed).split(idx):
            m = lgb.LGBMRegressor(**params).fit(Fc.iloc[idx[a]], target[idx[a]])
            pred[idx[b]] = m.predict(Fc.iloc[idx[b]])
            if len(miss):
                pred_miss += m.predict(Fc.iloc[miss]) / 5
        pred[miss] = pred_miss
        k = c[-1]
        out[f"EXT{k}_PRED"] = pred
        out[f"EXT{k}_FILL"] = np.where(has, target, pred)
        out[f"EXT{k}_RESID"] = np.where(has, target - pred, np.nan)
    fills = out[[f"EXT{k}_FILL" for k in "123"]]
    out["EXTF_MEAN"] = fills.mean(axis=1)
    out["EXTF_PROD"] = fills.prod(axis=1)
    out["EXTF_MIN"] = fills.min(axis=1)
    out["EXTF_W"] = 2 * out["EXT1_FILL"] + out["EXT2_FILL"] + 3 * out["EXT3_FILL"]
    return out

In [8]:
def _rate_stats(key, y_all, src):
    return pd.DataFrame({"k": key[src], "y": y_all[src]}).groupby("k")["y"].agg(["sum", "count"])

def _rates(key, st, dst, ks):
    kd = pd.Series(key[dst])
    s = kd.map(st["sum"]).fillna(0).values; c = kd.map(st["count"]).fillna(0).values
    return [(s + PRIOR * k) / (c + k) for k in ks]

def nested_rates(key, y_all, src, dst, ks=(30,), inner_seed=7):
    """Taxa de calote suavizada por chave; linhas de dst que estão em src recebem valor fora-da-parte."""
    out = [np.zeros(len(dst)) for _ in ks]
    in_src = np.isin(dst, src)
    if (~in_src).any():
        st = _rate_stats(key, y_all, src)
        for o, v in zip(out, _rates(key, st, dst[~in_src], ks)):
            o[~in_src] = v
    if in_src.any():
        d = dst[in_src]; tmp = [np.zeros(len(d)) for _ in ks]
        for ia, ib in StratifiedKFold(5, shuffle=True, random_state=inner_seed).split(d, y_all[d]):
            st = _rate_stats(key, y_all, d[ia])
            for t, v in zip(tmp, _rates(key, st, d[ib], ks)):
                t[ib] = v
        for o, t in zip(out, tmp):
            o[in_src] = t
    return out

class Supervised:
    """Acrescenta as colunas que dependem do TARGET para as linhas `dst`, usando só os rótulos de `src`."""
    def __init__(self, groups, tkeys, y_all):
        self.groups, self.tkeys, self.y_all = groups, tkeys, y_all
    def add(self, X_part, src, dst):
        X = X_part.copy(); y_all = self.y_all
        for name, g in self.groups.items():
            st = pd.DataFrame({"g": g[src], "y": y_all[src]}).groupby("g")["y"].agg(["sum", "count"])
            gs = pd.Series(g[dst])
            total = gs.map(st["sum"]).fillna(0).values; count = gs.map(st["count"]).fillna(0).values
            own = np.isin(dst, src); own_y = np.where(own, y_all[dst], 0)
            total, count = total - own_y, count - own
            X[f"P{name}_MATE_SUM"] = total
            X[f"P{name}_MATE_CNT"] = count
            X[f"P{name}_MATE_MEAN"] = np.where(count > 0, total / np.maximum(count, 1), np.nan)
        for name, key in self.tkeys.items():
            X[name + "_TE"] = nested_rates(key, y_all, src, dst, ks=(30,))[0]
        return X

In [9]:
LGB_BASE = dict(objective="binary", learning_rate=0.02, bagging_freq=1, deterministic=True,
                force_row_wise=True, verbose=-1, n_jobs=N_JOBS)
LGB_PARAMS = {
    "A": dict(LGB_BASE, num_leaves=35, min_child_samples=28, feature_fraction=0.157, bagging_fraction=0.98,
              reg_lambda=13.1, reg_alpha=0.92, min_split_gain=0.57, max_bin=127),
    "B": dict(LGB_BASE, num_leaves=9, min_child_samples=179, feature_fraction=0.44, bagging_fraction=0.74,
              reg_lambda=1.72, reg_alpha=0.23, min_split_gain=0.088, max_bin=63),
}
CAT_PARAMS = dict(learning_rate=0.04, depth=6, l2_leaf_reg=5, verbose=0, allow_writing_files=False,
                  thread_count=N_JOBS)

def make_model(kind, seed, n_trees=None, cat_cols=None):
    if kind in LGB_PARAMS:
        return lgb.LGBMClassifier(**LGB_PARAMS[kind], n_estimators=n_trees or 10000, random_state=seed)
    if kind == "cat":
        extra = dict(iterations=n_trees) if n_trees else dict(iterations=4000, eval_metric="AUC", od_type="Iter", od_wait=200)
        return CatBoostClassifier(**CAT_PARAMS, **extra, cat_features=cat_cols, random_seed=seed)
    raise ValueError(kind)

def fit_with_es(model, kind, Xtr, ytr, Xva, yva):
    if kind == "cat":
        model.fit(Xtr, ytr, eval_set=(Xva, yva), use_best_model=True)
        return model, model.get_best_iteration() + 1
    model.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="auc", callbacks=[lgb.early_stopping(200, verbose=False)])
    return model, model.best_iteration_

def to_rank(p):
    return (rankdata(p) - 0.5) / len(p)

In [10]:
def prepare(train, test, verbose=True):
    t0 = time.time()
    y = train["TARGET"].to_numpy(); n = len(train)
    raw = pd.concat([train.drop(columns="TARGET"), test], ignore_index=True)
    full = make_features(raw).drop(columns="SK_ID_CURR")          # o ID é só o número da linha: não entra
    cat_cols = full.select_dtypes(exclude="number").columns.tolist()
    groups = person_groups(raw)
    person = person_features(raw, groups)
    tariff = tariff_counts(raw)
    full_lgb = full.copy()
    for c in cat_cols:
        full_lgb[c + "_COUNT"] = full_lgb[c].map(full_lgb[c].value_counts())
        full_lgb[c] = full_lgb[c].astype("category").cat.codes
    full_lgb = pd.concat([full_lgb, person, tariff], axis=1)
    full_cat = full.copy()
    for c in cat_cols:
        full_cat[c] = full_cat[c].fillna("NA").astype(str)
    full_cat = pd.concat([full_cat, person, tariff], axis=1)
    plus = enhanced_features(raw, full_lgb)
    plus_cat = enhanced_features(raw, full_cat)
    ext = ext_aux_features(full_lgb, raw)
    sets = {"base": full_lgb,                                  # variáveis da v7
            "plusx": pd.concat([plus, ext], axis=1),           # + ampliadas + estimativas das EXT_SOURCE
            "catx": pd.concat([plus_cat, ext], axis=1)}        # idem, com as categorias em texto (CatBoost)
    y_all = np.concatenate([y, np.zeros(len(test))])
    ctx = dict(y=y, n=n, raw=raw, cat_cols=cat_cols, ids=test["SK_ID_CURR"].to_numpy(),
               sup=Supervised(groups, tariff_keys(raw), y_all),
               X={k: v.iloc[:n].reset_index(drop=True) for k, v in sets.items()},
               Xt={k: v.iloc[n:].reset_index(drop=True) for k, v in sets.items()})
    if verbose:
        print({k: v.shape[1] for k, v in sets.items()}, f"{time.time()-t0:.0f}s")
    return ctx

In [11]:
def run_cv(ctx, fset, kind, seeds=(42,), fold_seed=SEED, log=print):
    """Mesmas 5 partes para todos os componentes (fold_seed); `seeds` = sementes do modelo, cuja média é feita."""
    y, n, sup = ctx["y"], ctx["n"], ctx["sup"]
    X, Xt = ctx["X"][fset], ctx["Xt"][fset]
    idx_tr, idx_te = np.arange(n), np.arange(n, n + len(Xt))
    Xt_sup = sup.add(Xt, idx_tr, idx_te)
    oof = np.zeros(n); pte = np.zeros(len(Xt)); its = []
    folds = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=fold_seed).split(X, y))
    t0 = time.time()
    for f, (a, b) in enumerate(folds):
        Xa = sup.add(X.iloc[a], a, a); Xb = sup.add(X.iloc[b], a, b)
        for s in seeds:
            m = make_model(kind, s, cat_cols=ctx["cat_cols"])
            m, it = fit_with_es(m, kind, Xa, y[a], Xb, y[b]); its.append(it)
            oof[b] += m.predict_proba(Xb)[:, 1] / len(seeds)
            pte += m.predict_proba(Xt_sup)[:, 1] / (len(seeds) * N_FOLDS)
        log(f"  {fset}/{kind} fold {f}: AUC {roc_auc_score(y[b], oof[b]):.5f} ({time.time()-t0:.0f}s)")
    fold_id = np.zeros(n, int)
    for f, (a, b) in enumerate(folds):
        fold_id[b] = f
    return dict(oof=oof, test=pte, its=its, fold_id=fold_id, auc=roc_auc_score(y, oof))

def fit_full(ctx, fset, kind, n_trees, seeds=(42,)):
    y, n, sup = ctx["y"], ctx["n"], ctx["sup"]
    X, Xt = ctx["X"][fset], ctx["Xt"][fset]
    idx_tr, idx_te = np.arange(n), np.arange(n, n + len(Xt))
    Xall = sup.add(X, idx_tr, idx_tr); Xt_sup = sup.add(Xt, idx_tr, idx_te)
    p = np.zeros(len(Xt))
    for s in seeds:
        p += make_model(kind, s, n_trees=n_trees, cat_cols=ctx["cat_cols"]).fit(Xall, y).predict_proba(Xt_sup)[:, 1] / len(seeds)
    return p

In [12]:
def nn_available():
    try:
        import torch  # noqa
        return True
    except ImportError:
        return False

def nn_inputs(X_all_num, cat_codes, n_quantiles=1000):
    """Prepara a matriz numérica (quantis → normal, faltante → 0 + indicador). Sem TARGET."""
    from sklearn.preprocessing import QuantileTransformer
    Xn = X_all_num.replace([np.inf, -np.inf], np.nan)
    miss_cols = [c for c in Xn.columns if Xn[c].isna().mean() > 0.01]
    qt = QuantileTransformer(n_quantiles=n_quantiles, output_distribution="normal", subsample=10**9, random_state=0)
    Z = qt.fit_transform(Xn.to_numpy(dtype=np.float64))
    Z = np.nan_to_num(Z, nan=0.0).astype(np.float32)
    M = Xn[miss_cols].isna().to_numpy(np.float32)
    return np.hstack([Z, M]), cat_codes.astype(np.int64)

NN_PARAMS = dict(lr=5e-4, wd=1e-4, drop1=0.5, drop2=0.4, bs=1024, h1=512, h2=256)   # escolhidos na validação

def train_nn(Xn_tr, Xc_tr, y_tr, Xn_va, Xc_va, y_va, Xn_te_list, Xc_te_list, seed, cat_card,
             epochs=50, patience=6, n_epochs_fixed=None, log=None, hp=None):
    hp = dict(NN_PARAMS, **(hp or {}))
    import torch, torch.nn as nn
    torch.manual_seed(seed); np.random.seed(seed)
    torch.set_num_threads(N_JOBS); torch.use_deterministic_algorithms(True)
    d_num = Xn_tr.shape[1]

    class Net(nn.Module):
        def __init__(self):
            super().__init__()
            self.embs = nn.ModuleList([nn.Embedding(c + 2, min(16, (c + 2) // 2 + 1)) for c in cat_card])
            d_emb = sum(e.embedding_dim for e in self.embs)
            d_in = d_num + d_emb
            h1, h2 = hp["h1"], hp["h2"]
            self.mlp = nn.Sequential(nn.Linear(d_in, h1), nn.BatchNorm1d(h1), nn.ReLU(), nn.Dropout(hp["drop1"]),
                                     nn.Linear(h1, h2), nn.BatchNorm1d(h2), nn.ReLU(), nn.Dropout(hp["drop2"]),
                                     nn.Linear(h2, 1))
        def forward(self, xn, xc):
            e = [emb(xc[:, i] + 1) for i, emb in enumerate(self.embs)]
            return self.mlp(torch.cat([xn] + e, 1)).squeeze(1)

    net = Net(); opt = torch.optim.AdamW(net.parameters(), lr=hp["lr"], weight_decay=hp["wd"])
    lossf = nn.BCEWithLogitsLoss()
    tn = lambda a: torch.from_numpy(a)
    Xn_t, Xc_t, y_t = tn(Xn_tr), tn(Xc_tr), tn(y_tr.astype(np.float32))
    def predict(xn, xc):
        net.eval(); out = []
        with torch.no_grad():
            for i in range(0, len(xn), 8192):
                out.append(torch.sigmoid(net(tn(xn[i:i+8192]), tn(xc[i:i+8192]))).numpy())
        return np.concatenate(out)
    g = torch.Generator().manual_seed(seed)
    best, best_state, bad, best_ep = -1, None, 0, 0
    n_ep = n_epochs_fixed or epochs
    for ep in range(n_ep):
        net.train()
        perm = torch.randperm(len(Xn_t), generator=g)
        for i in range(0, len(perm), hp["bs"]):
            b = perm[i:i+hp["bs"]]
            opt.zero_grad(); loss = lossf(net(Xn_t[b], Xc_t[b]), y_t[b]); loss.backward(); opt.step()
        if n_epochs_fixed is None:
            auc = roc_auc_score(y_va, predict(Xn_va, Xc_va))
            if log: log(f"    época {ep+1}: AUC {auc:.5f}")
            if auc > best:
                best, bad, best_ep = auc, 0, ep + 1
                best_state = {k: v.clone() for k, v in net.state_dict().items()}
            else:
                bad += 1
                if bad >= patience:
                    break
    if best_state is not None:
        net.load_state_dict(best_state)
    pva = predict(Xn_va, Xc_va) if Xn_va is not None else None
    ptes = [predict(a, b) for a, b in zip(Xn_te_list, Xc_te_list)]
    return pva, ptes, best_ep

def _nn_split(ctx, Xs):
    cats = ctx["cat_cols"]
    num = [X.drop(columns=cats) for X in Xs]
    lens = [len(x) for x in num]
    Zall, Call = nn_inputs(pd.concat(num, ignore_index=True), pd.concat([X[cats] for X in Xs], ignore_index=True).to_numpy())
    out, s = [], 0
    for L in lens:
        out.append((Zall[s:s+L], Call[s:s+L])); s += L
    return out

def run_cv_nn(ctx, fset, seeds=(42,), fold_seed=SEED, log=print, hp=None, epochs=50, patience=6):
    y, n, sup = ctx["y"], ctx["n"], ctx["sup"]
    X, Xt = ctx["X"][fset], ctx["Xt"][fset]
    idx_tr, idx_te = np.arange(n), np.arange(n, n + len(Xt))
    Xt_sup = sup.add(Xt, idx_tr, idx_te)
    card = [int(max(X[c].max(), Xt[c].max())) + 1 for c in ctx["cat_cols"]]
    oof = np.zeros(n); pte = np.zeros(len(Xt)); its = []; t0 = time.time()
    folds = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=fold_seed).split(X, y))
    for f, (a, b) in enumerate(folds):
        Xa = sup.add(X.iloc[a], a, a); Xb = sup.add(X.iloc[b], a, b)
        (Za, Ca), (Zb, Cb), (Zt, Ct) = _nn_split(ctx, [Xa, Xb, Xt_sup])
        for s in seeds:
            pva, (pt,), ep = train_nn(Za, Ca, y[a], Zb, Cb, y[b], [Zt], [Ct], s, card, hp=hp,
                                      epochs=epochs, patience=patience)
            oof[b] += pva / len(seeds); pte += pt / (len(seeds) * N_FOLDS); its.append(ep)
        log(f"  NN {fset} fold {f}: AUC {roc_auc_score(y[b], oof[b]):.5f} épocas {its[-len(seeds):]} ({time.time()-t0:.0f}s)")
    return dict(oof=oof, test=pte, its=its, auc=roc_auc_score(y, oof))

def fit_full_nn(ctx, fset, n_epochs, seeds=(42,), hp=None):
    y, n, sup = ctx["y"], ctx["n"], ctx["sup"]
    X, Xt = ctx["X"][fset], ctx["Xt"][fset]
    idx_tr, idx_te = np.arange(n), np.arange(n, n + len(Xt))
    Xall = sup.add(X, idx_tr, idx_tr); Xt_sup = sup.add(Xt, idx_tr, idx_te)
    card = [int(max(X[c].max(), Xt[c].max())) + 1 for c in ctx["cat_cols"]]
    (Za, Ca), (Zt, Ct) = _nn_split(ctx, [Xall, Xt_sup])
    p = np.zeros(len(Xt))
    for s in seeds:
        _, (pt,), _ = train_nn(Za, Ca, y, None, None, None, [Zt], [Ct], s, card, n_epochs_fixed=n_epochs, hp=hp)
        p += pt / len(seeds)
    return p

Os campos `cv`, `weight` e `size` na lista original abaixo são o registro histórico da v17. As AUCs medidas, os pesos escolhidos e os tamanhos efetivamente usados pela v18 são calculados ou fixados nas células seguintes.

In [13]:
COMPONENTS = [{'label': 'LightGBM B + EXT auxiliares',
  'fset': 'plusx',
  'kind': 'B',
  'seeds': [42, 2024, 7],
  'size': 1646,
  'hp': None,
  'cv': 0.77231,
  'weight': 0.265},
 {'label': 'LightGBM B (variáveis da v7)',
  'fset': 'base',
  'kind': 'B',
  'seeds': [42, 2024, 7],
  'size': 1793,
  'hp': None,
  'cv': 0.77041,
  'weight': 0.051},
 {'label': 'Rede neural (MLP, 8 sementes)',
  'fset': 'plusx',
  'kind': 'nn',
  'seeds': [42, 2024, 7, 11, 99, 123, 5, 77],
  'size': 10,
  'hp': {'lr': 0.0005, 'wd': 0.0001, 'drop1': 0.5, 'drop2': 0.4, 'bs': 1024, 'h1': 512, 'h2': 256},
  'cv': 0.77039,
  'weight': 0.298},
 {'label': 'CatBoost + EXT auxiliares',
  'fset': 'catx',
  'kind': 'cat',
  'seeds': [42, 2024],
  'size': 1094,
  'hp': None,
  'cv': 0.77206,
  'weight': 0.386}]

for c in COMPONENTS:
    print(f"{c['label']:28s} peso {c['weight']:.3f}  sementes {c['seeds']}  tamanho {c['size']}")

LightGBM B + EXT auxiliares  peso 0.265  sementes [42, 2024, 7]  tamanho 1646
LightGBM B (variáveis da v7) peso 0.051  sementes [42, 2024, 7]  tamanho 1793
Rede neural (MLP, 8 sementes) peso 0.298  sementes [42, 2024, 7, 11, 99, 123, 5, 77]  tamanho 10
CatBoost + EXT auxiliares    peso 0.386  sementes [42, 2024]  tamanho 1094


## Novo componente: TabM periódico
Quantis ajustados apenas em Xa; categorias e auxiliares são as da v17. Não usa pricing.

In [14]:
"""TabM periódico da v16, com variáveis e Supervised da v17.

As funções da v17 não são redefinidas ou modificadas.
"""
import gc
import torch
from torch import nn
from sklearn.preprocessing import QuantileTransformer
from tabm import TabM
import rtdl_num_embeddings as num_embeddings

# Pool fixo em cada processo novo; os modelos ficam na CPU.
TORCH_INTEROP_THREADS = 2
if torch.get_num_interop_threads() != TORCH_INTEROP_THREADS:
    torch.set_num_interop_threads(TORCH_INTEROP_THREADS)
from threadpoolctl import threadpool_limits, threadpool_info
thread_limiter = threadpool_limits(limits=N_JOBS)
torch.set_num_threads(N_JOBS)
torch.use_deterministic_algorithms(True)

TABM_CONFIG = dict(k=8, n_blocks=2, d_block=128, dropout=0.15, arch_type="tabm")
TABM_AUX_COLS = [f"EXT{k}_{suffix}" for k in "123" for suffix in ["PRED", "FILL", "RESID"]]
TABM_AUX_COLS += ["EXTF_MEAN", "EXTF_PROD", "EXTF_MIN", "EXTF_W"]


def add_tabm_feature_set(ctx):
    # A v16 usava core=True: base (74 colunas) + 13 auxiliares.
    # Aqui todas as colunas vêm da prepare() original, sem recalcular features.
    for key in ["X", "Xt"]:
        ctx[key]["tabm_periodic"] = pd.concat(
            [ctx[key]["base"], ctx[key]["plusx"][TABM_AUX_COLS]], axis=1)
        assert "SK_ID_CURR" not in ctx[key]["tabm_periodic"]


def periodic_preprocess(ctx, A, others):
    cats = ctx["cat_cols"]
    nums = [c for c in A if c not in cats and A[c].nunique(dropna=True) > 1]
    qt = QuantileTransformer(n_quantiles=256, output_distribution="normal",
                             subsample=100000, random_state=42)
    values = A[nums].replace([np.inf, -np.inf], np.nan).to_numpy(dtype=np.float32)
    z = qt.fit_transform(values)
    keep = np.array([np.unique(v[np.isfinite(v)]).size > 1 for v in z.T])
    missing = [c for c in nums if A[c].isna().any()]
    # Cardinalidades sem TARGET; esquema categórico idêntico em todas as partes.
    X, Xt = ctx["X"]["tabm_periodic"], ctx["Xt"]["tabm_periodic"]
    cards = [int(max(X[c].max(), Xt[c].max(), 0)) + 2 for c in cats]
    cards += [2] * len(missing)

    def transform(frame):
        values = frame[nums].replace([np.inf, -np.inf], np.nan).to_numpy(dtype=np.float32)
        zn = np.nan_to_num(qt.transform(values), nan=0, posinf=4, neginf=-4)
        zn = zn.clip(-4, 4)[:, keep].astype(np.float32)
        zc = (frame[cats].to_numpy(dtype=np.int64) + 1).clip(min=0)
        if missing:
            zc = np.column_stack([zc, frame[missing].isna().to_numpy(dtype=np.int64)])
        return torch.from_numpy(zn), torch.from_numpy(zc)

    return transform(A), [transform(x) for x in others], cards


def periodic_model(n_num, cards):
    emb = num_embeddings.PeriodicEmbeddings(
        n_num, d_embedding=8, frequency_init_scale=0.1, lite=True)
    return TabM.make(n_num_features=n_num, num_embeddings=emb,
                     cat_cardinalities=cards, d_out=1, **TABM_CONFIG)


def periodic_predict(model, data):
    model.eval()
    pred = []
    with torch.no_grad():
        for start in range(0, len(data[0]), 2048):
            logits = model(data[0][start:start + 2048], data[1][start:start + 2048]).squeeze(-1)
            pred.append(logits.sigmoid().mean(dim=1).numpy())
    return np.concatenate(pred)


def train_periodic(pair, target, valid, y_valid, cards, seed, n_epochs_fixed=None, log=print):
    torch.set_num_threads(N_JOBS)
    torch.use_deterministic_algorithms(True)
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = periodic_model(pair[0].shape[1], cards)
    opt = torch.optim.AdamW(model.parameters(), lr=0.002, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="max", factor=0.5, patience=2, min_lr=1e-5)
    target = torch.tensor(target, dtype=torch.float32)
    best, best_ep, stale, state = -np.inf, 0, 0, None
    history = []
    for ep in range(1, (n_epochs_fixed or 40) + 1):
        ts = time.time()
        lr = float(opt.param_groups[0]["lr"])
        model.train()
        for ids in torch.randperm(len(target)).split(1024):
            opt.zero_grad(set_to_none=True)
            logits = model(pair[0][ids], pair[1][ids]).squeeze(-1)
            loss = nn.functional.binary_cross_entropy_with_logits(
                logits, target[ids, None].expand_as(logits))
            loss.backward()
            opt.step()
        if n_epochs_fixed is None:
            auc = float(roc_auc_score(y_valid, periodic_predict(model, valid)))
            scheduler.step(auc)
            history.append(dict(epoch=ep, auc=auc, lr=lr, seconds=time.time() - ts))
            if auc > best + 1e-6:
                best, best_ep, stale = auc, ep, 0
                state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                stale += 1
            log(f"    TabM época {ep}: AUC {auc:.7f}, melhor {best_ep}, {time.time()-ts:.1f}s")
            if stale >= 7:
                break
        else:
            # Sem validação no treino completo: LR constante, como os outros
            # componentes da v17; não consultar TARGET de validação/teste.
            log(f"    TabM treino completo época {ep}/{n_epochs_fixed}: {time.time()-ts:.1f}s")
    if state is not None:
        model.load_state_dict(state)
    return model, best_ep, history


def run_cv_periodic(ctx, log=print):
    y, n, sup = ctx["y"], ctx["n"], ctx["sup"]
    X = ctx["X"]["tabm_periodic"]
    oof = np.zeros(n)
    its, histories = [], []
    splits = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X, y))
    fold_id = np.empty(n, dtype=np.int8)
    for fold, (a, b) in enumerate(splits):
        fold_id[b] = fold
        assert not np.intersect1d(a, b).size
        Xa = sup.add(X.iloc[a], a, a)
        Xb = sup.add(X.iloc[b], a, b)
        pair, (valid,), cards = periodic_preprocess(ctx, Xa, [Xb])
        del Xa, Xb
        # Uma semente por parte, igual a experiment_v13.py: 42+100*fold.
        model, ep, history = train_periodic(pair, y[a], valid, y[b], cards, 42 + 100 * fold, log=log)
        oof[b] = periodic_predict(model, valid)
        its.append(ep)
        histories.append(history)
        log(f"TabM periódico parte {fold}: AUC {roc_auc_score(y[b], oof[b]):.7f}; época {ep}")
        del model, pair, valid
        gc.collect()
    return dict(oof=oof, its=its, histories=histories, fold_id=fold_id,
                auc=float(roc_auc_score(y, oof)))


def fit_full_periodic(ctx, n_epochs, log=print):
    y, n, sup = ctx["y"], ctx["n"], ctx["sup"]
    a = np.arange(n)
    b = np.arange(n, n + len(ctx["Xt"]["tabm_periodic"]))
    Xa = sup.add(ctx["X"]["tabm_periodic"], a, a)
    Xt = sup.add(ctx["Xt"]["tabm_periodic"], a, b)
    pair, (test_pair,), cards = periodic_preprocess(ctx, Xa, [Xt])
    model, _, _ = train_periodic(pair, y, None, None, cards, 42,
                                 n_epochs_fixed=int(n_epochs), log=log)
    result = periodic_predict(model, test_pair)
    del model, pair, test_pair
    gc.collect()
    return result


## Chamada de CV da MLP com TARGET isolado
As funções originais continuam intactas. Esta nova chamada reutiliza `Supervised.add`, `_nn_split` e `train_nn`, mas prepara também o teste com os rótulos do treino da parte. Assim, os quantis conjuntos não recebem colunas que dependam dos rótulos da validação. A mesma correção é aplicada à referência v17 e à candidata v18.

In [15]:
"""CV com isolamento completo do TARGET, sem editar funções da v17."""
def run_cv_nn_strict(ctx, fset, seeds=(42,), hp=None, folds_to_run=None, log=print):
    y, n, sup = ctx["y"], ctx["n"], ctx["sup"]
    X, Xt = ctx["X"][fset], ctx["Xt"][fset]
    idx_te = np.arange(n, n + len(Xt))
    cards = [int(max(X[c].max(), Xt[c].max())) + 1 for c in ctx["cat_cols"]]
    splits = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X, y))
    oof = np.zeros(n)
    its, per_fold = [], []
    for fold, (a, b) in enumerate(splits):
        if folds_to_run is not None and fold not in folds_to_run:
            continue
        Xa = sup.add(X.iloc[a], a, a)
        Xb = sup.add(X.iloc[b], a, b)
        # Diferença da chamada original: o teste entra nos quantis, então
        # suas colunas supervisionadas também precisam excluir y[b].
        Xt_fold = sup.add(Xt, a, idx_te)
        if fold == 0:
            changed = sup.y_all.copy()
            changed[b] = 1 - changed[b]
            check_sup = Supervised(sup.groups, sup.tkeys, changed)
            pd.testing.assert_frame_equal(Xa, check_sup.add(X.iloc[a], a, a))
            pd.testing.assert_frame_equal(Xb, check_sup.add(X.iloc[b], a, b))
            pd.testing.assert_frame_equal(Xt_fold, check_sup.add(Xt, a, idx_te))
            log("PASS: alterar todos os TARGET da validação não altera nenhum dos 3 frames usados nos quantis da MLP.")
        (Za, Ca), (Zb, Cb), _ = _nn_split(ctx, [Xa, Xb, Xt_fold])
        fold_its = []
        for seed in seeds:
            # As previsões de teste da CV não são necessárias para escolher
            # pesos ou avaliar OOF. O treino final usa fit_full_nn original.
            pva, _, ep = train_nn(Za, Ca, y[a], Zb, Cb, y[b], [], [], seed, cards, hp=hp)
            oof[b] += pva / len(seeds)
            its.append(ep)
            fold_its.append(ep)
            log(f"  MLP estrita parte {fold}, semente {seed}: época {ep}")
        auc = float(roc_auc_score(y[b], oof[b]))
        per_fold.append(dict(fold=fold, auc=auc, its=fold_its))
        log(f"MLP estrita parte {fold}: AUC {auc:.8f}")
    return dict(oof=oof, its=its, per_fold=per_fold,
                auc=float(roc_auc_score(y, oof)) if folds_to_run is None else None)


## Regra de seleção dos pesos
Grade única: TabM de 0% a 50%, de 2,5% em 2,5%; proporções da v17 preservadas. Empates favorecem o menor peso.

In [16]:
# A única busca nova é o peso do TabM: 21 valores, sem retunar modelos.
ALPHA_GRID = np.round(np.arange(0.0, 0.50001, 0.025), 3)
V17_WEIGHTS = np.array([0.265, 0.051, 0.298, 0.386], dtype=float)


def select_periodic_weight(y_select, squared_ranks_select):
    # Não recebe TARGET das partes 2–4. Os percentis são os mesmos da tabela.
    baseline = squared_ranks_select[:, :4] @ V17_WEIGHTS
    scores = []
    for alpha in ALPHA_GRID:
        p = (1 - alpha) * baseline + alpha * squared_ranks_select[:, 4]
        scores.append(dict(alpha=float(alpha), auc=float(roc_auc_score(y_select, p))))
    best = max(scores, key=lambda r: r["auc"])  # empate: primeiro/menor peso
    return best["alpha"], scores


def compare_v17_v18(y, oofs):
    splits = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
                  .split(np.zeros(len(y)), y))
    fold_id = np.empty(len(y), dtype=np.int8)
    for fold, (_, b) in enumerate(splits):
        fold_id[b] = fold
    selection = fold_id < 2
    ranks = np.column_stack([to_rank(p) ** 2 for p in oofs])
    alpha, search = select_periodic_weight(y[selection], ranks[selection])
    weights = np.r_[(1 - alpha) * V17_WEIGHTS, alpha]
    v17 = ranks[:, :4] @ V17_WEIGHTS
    v18 = ranks @ weights if alpha > 0 else v17.copy()
    rows = []
    masks = [(f"parte_{i}", fold_id == i) for i in range(5)]
    masks += [("partes_0_1", selection), ("partes_2_4", ~selection),
              ("total", np.ones(len(y), bool))]
    for label, mask in masks:
        a = float(roc_auc_score(y[mask], v17[mask]))
        b = float(roc_auc_score(y[mask], v18[mask]))
        rows.append(dict(parte=label, linhas=int(mask.sum()), v17=a, v18=b, delta=b-a))
    by_name = {r["parte"]: r for r in rows}
    improved = (by_name["partes_2_4"]["delta"] > 0 and by_name["total"]["delta"] > 0)
    return dict(alpha=alpha, weights=weights.tolist(), search=search,
                table=rows, improved=bool(improved),
                baseline_note="v17 recalculada com isolamento do TARGET na chamada da MLP; "
                              "as 24 funções/classes originais estão intactas.",
                selection_note="Mesmos percentis globais ao quadrado da v17; "
                               "só TARGET das partes 0–1 escolhe o peso. "
                               "Nenhum ajuste após consultar partes 2–4.")


In [17]:
import scipy, sklearn, catboost, tabm, rtdl_num_embeddings
LOADED_VERSIONS = {"numpy": np.__version__, "pandas": pd.__version__,
                   "scipy": scipy.__version__, "scikit-learn": sklearn.__version__,
                   "lightgbm": lgb.__version__, "catboost": catboost.__version__,
                   "torch": str(torch.__version__)}
assert all(PINNED[k] == v.split("+")[0] for k, v in LOADED_VERSIONS.items()), (
    "Reinicie o runtime: versões carregadas diferem das instaladas.", LOADED_VERSIONS)


In [18]:
def v20_baseline_fold(ctx, fold, split, log=print):
    """Same learners and seeds as v18, no unused test prediction overhead."""
    a, b = split
    y, sup = ctx["y"], ctx["sup"]
    predictions, iterations = [], []
    for component in COMPONENTS:
        started = time.time()
        if component["kind"] == "nn":
            result = run_cv_nn_strict(ctx, component["fset"], seeds=tuple(component["seeds"]),
                                      hp=component["hp"], folds_to_run=[fold], log=log)
            pred, its = result["oof"][b], result["its"]
        else:
            X = ctx["X"][component["fset"]]
            xa, xb = sup.add(X.iloc[a], a, a), sup.add(X.iloc[b], a, b)
            pred, its = np.zeros(len(b)), []
            for seed in component["seeds"]:
                model = make_model(component["kind"], seed, cat_cols=ctx["cat_cols"])
                model, iteration = fit_with_es(model, component["kind"], xa, y[a], xb, y[b])
                pred += model.predict_proba(xb)[:, 1] / len(component["seeds"])
                its.append(int(iteration))
                log(f"BASELINE parte {fold}: {component['label']} seed {seed} concluída, tamanho {iteration}", flush=True)
        predictions.append(pred)
        iterations.append(its)
        log(f"BASELINE parte {fold}: {component['label']}, "
            f"AUC {roc_auc_score(y[b], pred):.8f}, {time.time()-started:.1f}s", flush=True)
    X = ctx["X"]["tabm_periodic"]
    xa, xb = sup.add(X.iloc[a], a, a), sup.add(X.iloc[b], a, b)
    pair, (valid,), cards = periodic_preprocess(ctx, xa, [xb])
    model, epoch, _ = train_periodic(pair, y[a], valid, y[b], cards, 42 + 100 * fold,
                                    log=lambda message: None)
    predictions.append(periodic_predict(model, valid))
    iterations.append([int(epoch)])
    log(f"BASELINE TabM parte {fold}: AUC {roc_auc_score(y[b], predictions[-1]):.8f}, época {epoch}", flush=True)
    del model, pair, valid
    gc.collect()
    return np.column_stack(predictions), iterations

In [19]:
"""v20: bounded new-model search and rebalancing, selection labels 0–1 only."""
import joblib

V20_BASE_WEIGHTS = np.array([0.19875, 0.03825, 0.22349999999999998, 0.2895, 0.25])
V20_CANDIDATES = [
    dict(label="lgb_bins255", kind="lgb_resolution", max_bin=255),
    dict(label="lgb_bins1023", kind="lgb_resolution", max_bin=1023),
    dict(label="tabm_pair_half", kind="tabm_pairwise", pair_weight=0.5),
    dict(label="tabm_pair_full", kind="tabm_pairwise", pair_weight=1.0),
]
V20_CV_WORKERS = 2


def v20_initialize_worker():
    import os
    for name in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"]:
        os.environ[name] = "2"
    # Import before deserializing LightGBM functions: supplies libomp on macOS.
    import torch
    torch.set_num_threads(2)
    if torch.get_num_interop_threads() != 2:
        torch.set_num_interop_threads(2)
    torch.use_deterministic_algorithms(True)
    from threadpoolctl import threadpool_limits
    global _v20_worker_limit
    _v20_worker_limit = threadpool_limits(limits=2)


def v20_train_pairwise(pair, target, valid, y_valid, cards, seed, pair_weight, n_epochs_fixed=None):
    torch.set_num_threads(2)
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.use_deterministic_algorithms(True)
    model = periodic_model(pair[0].shape[1], cards)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.002, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2, min_lr=1e-5)
    labels = torch.tensor(target, dtype=torch.float32)
    # Pair sampling has its own fixed stream; it does not change dropout/shuffle.
    generator = torch.Generator().manual_seed(seed + 8000)
    best, best_epoch, stale, state = -np.inf, 0, 0, None
    history = []
    for epoch in range(1, (n_epochs_fixed or 40) + 1):
        model.train()
        for ids in torch.randperm(len(labels)).split(1024):
            optimizer.zero_grad(set_to_none=True)
            logits = model(pair[0][ids], pair[1][ids]).squeeze(-1)
            batch_y = labels[ids]
            positive = torch.where(batch_y == 1)[0]
            negative = torch.where(batch_y == 0)[0]
            bce = nn.functional.binary_cross_entropy_with_logits(logits, batch_y[:, None].expand_as(logits))
            if len(positive) and len(negative):
                pi = positive[torch.randint(len(positive), (4096,), generator=generator)]
                ni = negative[torch.randint(len(negative), (4096,), generator=generator)]
                pair_loss = nn.functional.softplus(logits[ni] - logits[pi]).mean()
                loss = (1 - pair_weight) * bce + pair_weight * pair_loss
            else:
                # No valid pairs in a rare last batch: no update of the model.
                continue
            loss.backward()
            optimizer.step()
        if n_epochs_fixed is None:
            auc = float(roc_auc_score(y_valid, periodic_predict(model, valid)))
            scheduler.step(auc)
            history.append(dict(epoch=epoch, auc=auc, lr=float(optimizer.param_groups[0]["lr"])))
            if auc > best + 1e-6:
                best, best_epoch, stale = auc, epoch, 0
                state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                stale += 1
            if stale >= 7:
                break
    if state is not None:
        model.load_state_dict(state)
    return model, int(best_epoch), history


def v20_candidate_fold(ctx, fold, split, configuration):
    a, b = split
    fset = "plusx" if configuration["kind"] == "lgb_resolution" else "tabm_periodic"
    X, sup, y = ctx["X"][fset], ctx["sup"], ctx["y"]
    xa, xb = sup.add(X.iloc[a], a, a), sup.add(X.iloc[b], a, b)
    altered = sup.y_all.copy()
    altered[b] = 1 - altered[b]
    isolated = Supervised(sup.groups, sup.tkeys, altered)
    pd.testing.assert_frame_equal(xa, isolated.add(X.iloc[a], a, a))
    pd.testing.assert_frame_equal(xb, isolated.add(X.iloc[b], a, b))
    started = time.time()
    if configuration["kind"] == "lgb_resolution":
        parameters = dict(LGB_PARAMS["B"], max_bin=configuration["max_bin"])
        model = lgb.LGBMClassifier(**parameters, n_estimators=10000, random_state=42)
        model, iteration = fit_with_es(model, "B", xa, y[a], xb, y[b])
        p = model.predict_proba(xb)[:, 1]
    else:
        pair, (valid,), cards = periodic_preprocess(ctx, xa, [xb])
        model, iteration, _ = v20_train_pairwise(pair, y[a], valid, y[b], cards,
                                                 42 + 100 * fold, configuration["pair_weight"])
        p = periodic_predict(model, valid)
    print(f"NOVO parte {fold}: {configuration['label']}, AUC {roc_auc_score(y[b], p):.8f}, "
          f"tamanho {iteration}, {time.time()-started:.1f}s", flush=True)
    return p, int(iteration)


def v20_fold_job(ctx, fold, split, candidate_indices):
    # All returned predictions were fitted in this job; nothing read from disk.
    started = time.time()
    print(f"PARTE {fold} iniciada, candidatos {candidate_indices}", flush=True)
    baseline, iterations = v20_baseline_fold(ctx, fold, split)
    predictions, candidate_iterations = {}, {}
    for index in candidate_indices:
        predictions[index], candidate_iterations[index] = v20_candidate_fold(ctx, fold, split, V20_CANDIDATES[index])
    print(f"PARTE {fold} concluída, {time.time()-started:.1f}s", flush=True)
    return dict(fold=fold, baseline=baseline, iterations=iterations, candidates=predictions,
                candidate_iterations=candidate_iterations, seconds=time.time()-started,
                threads=2, torch_interop_pool=torch.get_num_interop_threads(), threadpools=threadpool_info())


def v20_weights(y_selection, squared_ranks, has_candidate):
    # Original proportions are a starting point; mass transfers stay within
    # +/-0.10 of each original weight. No regression/stacking model is fitted.
    reference = np.r_[V20_BASE_WEIGHTS, 0.] if has_candidate else V20_BASE_WEIGHTS.copy()
    lower = np.maximum(reference - 0.10, 0)
    upper = reference + 0.10
    if has_candidate:
        upper[-1] = 0.30
    starts = [reference]
    if has_candidate:
        starts += [np.r_[(1-alpha)*V20_BASE_WEIGHTS, alpha] for alpha in np.arange(0.025, 0.30001, 0.025)]
    evaluations = 0

    def evaluate(weights):
        nonlocal evaluations
        evaluations += 1
        return float(roc_auc_score(y_selection, squared_ranks @ weights))

    scores = [evaluate(w) for w in starts]
    best_index = int(np.argmax(scores))
    weights, score = starts[best_index].copy(), scores[best_index]
    for step in [0.025, 0.01]:
        for _ in range(4):
            move, next_score = None, score
            for source in range(len(weights)):
                for destination in range(len(weights)):
                    if source == destination or weights[source]-step < lower[source]-1e-12 or weights[destination]+step > upper[destination]+1e-12:
                        continue
                    proposed = weights.copy()
                    proposed[source] -= step
                    proposed[destination] += step
                    value = evaluate(proposed)
                    if value > next_score + 1e-12:
                        next_score, move = value, proposed
            if move is None:
                break
            weights, score = move, next_score
    weights[np.abs(weights) < 1e-12] = 0
    assert np.all(weights >= 0) and abs(weights.sum()-1) < 1e-10
    return dict(weights=weights.tolist(), auc=score, evaluations=evaluations)


def v20_run(ctx, directory):
    directory.mkdir(parents=True, exist_ok=True)
    started = time.time()
    y, n = ctx["y"], ctx["n"]
    splits = list(StratifiedKFold(5, shuffle=True, random_state=42).split(np.zeros(n), y))
    fold_ids = np.empty(n, dtype=np.int8)
    for fold, (_, b) in enumerate(splits):
        fold_ids[b] = fold
    selection = fold_ids < 2
    baseline_oof = np.full((n, 5), np.nan)
    candidate_oof = np.full((n, len(V20_CANDIDATES)), np.nan)
    iterations = [[] for _ in range(5)]
    candidate_iterations = [[] for _ in V20_CANDIDATES]
    report = dict(status="running", confirmed=False, csv_sha256=None, input_sha256=input_hashes,
                  configurations=V20_CANDIDATES, versions=PINNED, python=sys.version, platform=platform.platform(),
                  cv_workers=V20_CV_WORKERS, threads_per_model=2, fold_results=[],
                  baseline_weights=V20_BASE_WEIGHTS.tolist(), selection_folds=[0,1], confirmation_folds=[2,3,4])

    def save():
        report["seconds"] = time.time()-started
        (directory / "experiment_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n")

    def receive(results):
        for result in results:
            fold = result["fold"]
            b = splits[fold][1]
            baseline_oof[b] = result["baseline"]
            for dst, values in zip(iterations, result["iterations"]):
                dst.extend(values)
            for index, values in result["candidates"].items():
                candidate_oof[b, index] = values
                candidate_iterations[index].append(result["candidate_iterations"][index])
            report["fold_results"].append({k: result[k] for k in ["fold", "seconds", "threads", "torch_interop_pool", "threadpools"]})
            save()

    save()
    # Two independent clean worker processes, two threads per fitted model.
    compact_ctx = {k:v for k,v in ctx.items() if k != "raw"}
    with joblib.parallel_config(backend="loky", n_jobs=V20_CV_WORKERS, inner_max_num_threads=2,
                                initializer=v20_initialize_worker):
        receive(joblib.Parallel(max_nbytes=None)(joblib.delayed(v20_fold_job)(compact_ctx, fold, splits[fold], list(range(4)))
                                                 for fold in [0,1]))
        ranks = np.column_stack([to_rank(p)**2 for p in baseline_oof[selection].T])
        base_score = float(roc_auc_score(y[selection], ranks @ V20_BASE_WEIGHTS))
        searches = [dict(candidate=None, **v20_weights(y[selection], ranks, False))]
        for index in range(len(V20_CANDIDATES)):
            r = np.column_stack([ranks, to_rank(candidate_oof[selection,index])**2])
            searches.append(dict(candidate=index, **v20_weights(y[selection], r, True)))
        chosen = max(searches, key=lambda row:row["auc"])
        report.update(screening=searches, screening_baseline_auc=base_score, chosen_family=chosen["candidate"])
        print("TRIAGEM", json.dumps(dict(baseline=base_score, candidates=searches)), flush=True)
        if chosen["auc"] <= base_score + 1e-12:
            report.update(status="rejected_in_screening", reason="No strategy improves folds 0–1; no new CSV.")
            save()
            return report
        selected_candidate = chosen["candidate"]
        qualified = [] if selected_candidate is None else [selected_candidate]
        report["status"] = "completing_oof"
        save()
        receive(joblib.Parallel(max_nbytes=None)(joblib.delayed(v20_fold_job)(compact_ctx, fold, splits[fold], qualified)
                                                 for fold in [2,3,4]))
    # Ranks now use all OOF rows, exactly as v18. Only labels 0–1 tune weights.
    ranks = np.column_stack([to_rank(p)**2 for p in baseline_oof.T])
    candidate_ranks = ranks if selected_candidate is None else np.column_stack([ranks, to_rank(candidate_oof[:,selected_candidate])**2])
    frozen = v20_weights(y[selection], candidate_ranks[selection], selected_candidate is not None)
    report.update(status="recipe_frozen_before_confirmation", weights=frozen["weights"], selection_auc=frozen["auc"],
                  selected_candidate=None if selected_candidate is None else V20_CANDIDATES[selected_candidate],
                  baseline_iterations=iterations, baseline_sizes=[max(1,round(float(np.mean(it))*1.15)) for it in iterations],
                  candidate_iterations=None if selected_candidate is None else candidate_iterations[selected_candidate],
                  candidate_size=None if selected_candidate is None else max(1,round(float(np.mean(candidate_iterations[selected_candidate]))*1.15)))
    (directory / "frozen_selection.json").write_text(json.dumps(report,ensure_ascii=False,indent=2)+"\n")
    save()
    base, new = ranks @ V20_BASE_WEIGHTS, candidate_ranks @ np.array(frozen["weights"])
    masks = [(f"parte_{fold}", fold_ids==fold) for fold in range(5)]
    masks += [("partes_0_1",selection),("partes_2_4",~selection),("total",np.ones(n,bool))]
    table = []
    for label, mask in masks:
        a, b = float(roc_auc_score(y[mask],base[mask])),float(roc_auc_score(y[mask],new[mask]))
        table.append(dict(parte=label,linhas=int(mask.sum()),v18=a,v20=b,delta=b-a))
    by_name = {r["parte"]:r for r in table}
    confirmed = by_name["partes_2_4"]["delta"]>0 and by_name["total"]["delta"]>0
    report.update(table=table,confirmed=bool(confirmed),status="gain_confirmed" if confirmed else "rejected_in_confirmation")
    save()
    print(pd.DataFrame(table).to_string(index=False,float_format=lambda v:f"{v:.8f}"),flush=True)
    return report


In [20]:
"""Predeclared v21 candidates; train from input, select on folds 0–1 only."""

V21_WEIGHTS = np.array([0.17396875, 0.00655625, 0.23438749999999997,
                        0.19383749999999997, 0.21625, 0.17500000000000002])
V21_CONFIGS = [
    dict(label="cat_ordered_depth5", kind="cat_ordered", depth=5),
    dict(label="cat_ordered_depth6", kind="cat_ordered", depth=6),
    dict(label="lgb_extra_trees", kind="lgb_extra"),
]
V21_ALPHAS = np.round(np.arange(0, 0.30001, 0.025), 3)


def v21_candidate_model(configuration, size=None, cat_cols=None):
    if configuration["kind"] == "cat_ordered":
        parameters = dict(CAT_PARAMS, depth=configuration["depth"], boosting_type="Ordered",
                          random_seed=42, cat_features=cat_cols)
        if size is None:
            parameters.update(iterations=4000, eval_metric="AUC", od_type="Iter", od_wait=200)
        else:
            parameters["iterations"] = int(size)
        return CatBoostClassifier(**parameters)
    parameters = dict(LGB_PARAMS["B"], extra_trees=True, extra_seed=42,
                      n_estimators=int(size) if size is not None else 10000, random_state=42)
    return lgb.LGBMClassifier(**parameters)


def v21_fit_candidate(ctx, fold, split, configuration):
    a, b = split
    fset = "catx" if configuration["kind"] == "cat_ordered" else "plusx"
    X, sup, y = ctx["X"][fset], ctx["sup"], ctx["y"]
    xa, xb = sup.add(X.iloc[a], a, a), sup.add(X.iloc[b], a, b)
    altered = sup.y_all.copy()
    altered[b] = 1 - altered[b]
    isolated = Supervised(sup.groups, sup.tkeys, altered)
    pd.testing.assert_frame_equal(xa, isolated.add(X.iloc[a], a, a))
    pd.testing.assert_frame_equal(xb, isolated.add(X.iloc[b], a, b))
    assert "SK_ID_CURR" not in xa and "TARGET" not in xa
    started = time.time()
    print(f"NOVO parte {fold}: {configuration['label']} iniciado", flush=True)
    model = v21_candidate_model(configuration, cat_cols=ctx["cat_cols"])
    if configuration["kind"] == "cat_ordered":
        model.fit(xa, y[a], eval_set=(xb, y[b]), use_best_model=True, verbose=500)
        iteration = model.get_best_iteration()+1
    else:
        model, iteration = fit_with_es(model, "B", xa, y[a], xb, y[b])
    p = model.predict_proba(xb)[:, 1]
    print(f"NOVO parte {fold}: {configuration['label']}, AUC {roc_auc_score(y[b],p):.8f}, "
          f"tamanho {iteration}, {time.time()-started:.1f}s", flush=True)
    return p, int(iteration)


def v21_job(ctx, fold, split, candidates):
    started = time.time()
    # Reuse original v20 learners in RAM, refitting all six in this new run.
    print(f"PARTE {fold} iniciada; candidatos {candidates}", flush=True)
    baseline, iterations = v20_baseline_fold(ctx, fold, split)
    pair_pred, pair_size = v20_candidate_fold(ctx, fold, split,
                                            dict(label="tabm_pair_full",kind="tabm_pairwise",pair_weight=1.0))
    baseline = np.column_stack([baseline, pair_pred])
    iterations.append([pair_size])
    predictions, sizes = {}, {}
    for index in candidates:
        predictions[index], sizes[index] = v21_fit_candidate(ctx, fold, split, V21_CONFIGS[index])
    print(f"PARTE {fold} concluída; {time.time()-started:.1f}s", flush=True)
    return dict(fold=fold, baseline=baseline, baseline_iterations=iterations,
                predictions=predictions, candidate_sizes=sizes, seconds=time.time()-started,
                threads=2, threadpools=threadpool_info(), torch_interop_pool=torch.get_num_interop_threads())


def v21_choose(target, baseline, candidate):
    results = []
    for alpha in V21_ALPHAS:
        p = (1-alpha)*baseline + alpha*candidate
        results.append(dict(alpha=float(alpha), auc=float(roc_auc_score(target,p))))
    # Stable max resolves exact ties in favor of zero/smaller new weight.
    return max(results, key=lambda r:r["auc"]), results


def v21_run(ctx, directory):
    directory.mkdir(parents=True, exist_ok=True)
    started = time.time()
    y, n = ctx["y"], ctx["n"]
    splits = list(StratifiedKFold(n_splits=5,shuffle=True,random_state=42).split(np.zeros(n),y))
    fold_ids = np.empty(n, dtype=np.int8)
    for fold, (_, b) in enumerate(splits):
        fold_ids[b] = fold
    selection = fold_ids < 2
    baseline_oof = np.full((n,6),np.nan)
    new_oof = np.full((n,3),np.nan)
    baseline_iterations = [[] for _ in range(6)]
    new_iterations = [[] for _ in range(3)]
    report = dict(status="running",confirmed=False,csv_sha256=None,
                  configurations=V21_CONFIGS,alpha_grid=V21_ALPHAS.tolist(),
                  baseline_weights=V21_WEIGHTS.tolist(),input_sha256=input_hashes,
                  versions=PINNED,exact_installed_versions={p:importlib.metadata.version(p) for p in PINNED},
                  python=sys.version,platform=platform.platform(),fold_results=[],selection_folds=[0,1],
                  confirmation_folds=[2,3,4],cv_workers=2,threads_per_model=2,
                  source_note="All six baseline learners and all candidates refit from the two input CSVs; no predictions or models loaded.")

    def save():
        report["seconds"] = time.time()-started
        (directory/"experiment_report.json").write_text(json.dumps(report,ensure_ascii=False,indent=2)+"\n")

    def receive(results):
        for r in results:
            b = splits[r["fold"]][1]
            baseline_oof[b] = r["baseline"]
            for dst, values in zip(baseline_iterations,r["baseline_iterations"]):
                dst.extend(values)
            for index,p in r["predictions"].items():
                new_oof[b,index] = p
                new_iterations[index].append(r["candidate_sizes"][index])
            report["fold_results"].append({k:r[k] for k in ["fold","seconds","threads","threadpools","torch_interop_pool"]})
            save()

    save()
    compact = {k:v for k,v in ctx.items() if k!="raw"}
    with joblib.parallel_config(backend="loky",n_jobs=2,inner_max_num_threads=2,initializer=v20_initialize_worker):
        receive(joblib.Parallel(max_nbytes=None)(joblib.delayed(v21_job)(compact,f,splits[f],[0,1,2]) for f in [0,1]))
        ranks = np.column_stack([to_rank(p)**2 for p in baseline_oof[selection].T])
        base = ranks@V21_WEIGHTS
        baseline_auc = float(roc_auc_score(y[selection],base))
        searches=[]
        for index in range(3):
            result,grid = v21_choose(y[selection],base,to_rank(new_oof[selection,index])**2)
            searches.append(dict(candidate=index,**result,grid=grid))
        chosen=max(searches,key=lambda r:r["auc"])
        report.update(screening=searches,screening_baseline_auc=baseline_auc)
        print("TRIAGEM",json.dumps(dict(baseline=baseline_auc,searches=searches)),flush=True)
        if chosen["alpha"]==0 or chosen["auc"]<=baseline_auc+1e-12:
            report.update(status="rejected_in_screening",reason="None of the three candidates improves folds 0–1; no new CSV.")
            save()
            return report
        index=chosen["candidate"]
        report.update(status="completing_oof",chosen_family=index)
        save()
        receive(joblib.Parallel(max_nbytes=None)(joblib.delayed(v21_job)(compact,f,splits[f],[index]) for f in [2,3,4]))
    assert np.isfinite(baseline_oof).all() and np.isfinite(new_oof[:,index]).all()
    ranks = np.column_stack([to_rank(p)**2 for p in baseline_oof.T])
    base = ranks@V21_WEIGHTS
    candidate = to_rank(new_oof[:,index])**2
    chosen,grid = v21_choose(y[selection],base[selection],candidate[selection])
    alpha=chosen["alpha"]
    weights=np.r_[(1-alpha)*V21_WEIGHTS,alpha]
    report.update(status="recipe_frozen_before_confirmation",alpha=alpha,weights=weights.tolist(),
                  selected_candidate=V21_CONFIGS[index],selection_auc=chosen["auc"],final_selection_grid=grid,
                  baseline_iterations=baseline_iterations,candidate_iterations=new_iterations[index],
                  baseline_sizes=[max(1,round(float(np.mean(it))*1.15)) for it in baseline_iterations],
                  candidate_size=max(1,round(float(np.mean(new_iterations[index]))*1.15)))
    (directory/"frozen_selection.json").write_text(json.dumps(report,ensure_ascii=False,indent=2)+"\n")
    save()
    new=(1-alpha)*base+alpha*candidate
    masks=[(f"parte_{f}",fold_ids==f) for f in range(5)]
    masks += [("partes_0_1",selection),("partes_2_4",~selection),("total",np.ones(n,bool))]
    table=[]
    for label,mask in masks:
        old_auc=float(roc_auc_score(y[mask],base[mask]))
        new_auc=float(roc_auc_score(y[mask],new[mask]))
        table.append(dict(parte=label,linhas=int(mask.sum()),v20=old_auc,v21=new_auc,delta=new_auc-old_auc))
    by_name={r["parte"]:r for r in table}
    assert abs(by_name["total"]["v20"]-0.7740056552208389)<1e-12,"Reference v20 not reproduced; investigate before generating a CSV."
    assert abs(by_name["partes_2_4"]["v20"]-0.7747163432212283)<1e-12
    confirmed=alpha>0 and by_name["partes_2_4"]["delta"]>0 and by_name["total"]["delta"]>0
    report.update(table=table,confirmed=bool(confirmed),status="gain_confirmed" if confirmed else "rejected_in_confirmation")
    save()
    print(pd.DataFrame(table).to_string(index=False,float_format=lambda v:f"{v:.8f}"),flush=True)
    return report


In [21]:
"""One frozen PyTorch factorization machine; selection labels 0–1 only.

All new supervised preprocessing and epoch selection exclude the outer
evaluation labels. Historical v21 learners are refitted without modifications.
Saved predictions are evidence only: this module never loads any prediction.
"""
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import OneHotEncoder
from scipy.stats import spearmanr

V24_WEIGHTS = np.array([0.156571875, 0.005900625, 0.21094875,
                        0.17445374999999996, 0.194625, 0.15750000000000003, 0.1])
V24_LABELS = ["LightGBM B + EXT", "LightGBM B v7", "MLP 8 sementes",
              "CatBoost Plain + EXT", "TabM periódico", "TabM por pares",
              "CatBoost Ordered depth 5"]
V24_PROTOCOL = dict(
    version="v24", hypothesis="PyTorch second-order factorization machine",
    deadline="2026-11-14", seed=42, threads_per_model=2, cv_workers=2,
    folds=dict(n_splits=5, shuffle=True, random_state=42),
    selection_folds=[0, 1], confirmation_folds=[2, 3, 4],
    alpha_grid=[0.0, 0.05, 0.10, 0.15, 0.20], tie_break="smaller_alpha",
    baseline_weights=V24_WEIGHTS.tolist(),
    baseline_final_sizes=[1646, 1793, 11, 1103, 14, 11, 1089],
    fm=dict(feature_set="plusx", rank=16, linear=True, loss="unweighted_BCE",
            optimizer="AdamW", lr=0.002, weight_decay=0.001, batch_size=2048,
            max_epochs=40, patience=6, min_delta=0.000001,
            inner_fraction=0.15, inner_random_state=42,
            factor_init_std=0.01, linear_init="zero",
            bias_init="logit_of_training_prevalence", model_seed=42,
            quantiles=256, quantile_distribution="normal", clip=4.0,
            categorical="dense_one_hot_unknown_ignore",
            missing="zero_numeric_plus_training_missing_indicators",
            constant_columns="drop_by_training_only",
            epoch_selection="inner_AUC_then_refit_all_outer_training_from_scratch",
            final_epochs="rounded_mean_of_outer_inner_selected_epochs_no_multiplier"),
    strata=dict(key="round(AMT_CREDIT / AMT_ANNUITY, 2)", decimals=2,
                count_source="outer_training_only_excluding_evaluated_fold",
                bins=[[0, 9], [10, 49], [50, 199], [200, None]],
                names=["raras_0_9", "pouco_frequentes_10_49", "comuns_50_199", "muito_frequentes_200_mais"],
                invalid="missing_nonfinite_or_nonpositive_credit_or_annuity",
                use="diagnostics_only_never_feature_weight_or_prediction_rule"),
    rank_procedure="screen_on_0_1_ranks_then_if_pass_reselect_same_grid_with_global_ranks_using_only_labels_0_1",
    screening="stop_if_best_alpha_zero_or_no_positive_gain_on_0_1",
    promotion=dict(min_delta_2_4=0.00010, total_delta_strictly_positive=True,
                   reference_2_4=0.7747809092432562, reference_total=0.7740567649584406),
    final_limit_seconds=3600, no_prediction_or_model_loading=True,
    note="Inherited v21 unlabeled train+test preprocessing and v21 early stopping are preserved for exact comparison. New FM epoch selection uses an inner split.")


def v24_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False)+"\n")


def v24_frozen_protocol(directory):
    """Write before any model fit; a rerun must use identical definitions."""
    path = Path(directory)/"frozen_protocol.json"
    if path.exists():
        assert json.loads(path.read_text()) == V24_PROTOCOL, "Protocol changed after freezing."
    else:
        v24_json(path, V24_PROTOCOL)
    return hashlib.sha256(path.read_bytes()).hexdigest()


def v24_strata(train, splits):
    credit = train.AMT_CREDIT.to_numpy(dtype=float)
    annuity = train.AMT_ANNUITY.to_numpy(dtype=float)
    valid = np.isfinite(credit) & np.isfinite(annuity) & (credit > 0) & (annuity > 0)
    ratio = np.full(len(train), np.nan)
    ratio[valid] = np.round(credit[valid]/annuity[valid], 2)
    valid &= np.isfinite(ratio)
    frequency = np.full(len(train), -1, dtype=np.int64)
    labels = np.full(len(train), "ausentes_invalidas", dtype=object)
    for a, b in splits:
        counts = pd.Series(ratio[a][valid[a]]).value_counts()
        dst = b[valid[b]]
        frequency[dst] = pd.Series(ratio[dst]).map(counts).fillna(0).to_numpy(dtype=np.int64)
        for (lo, hi), name in zip(V24_PROTOCOL["strata"]["bins"], V24_PROTOCOL["strata"]["names"]):
            chosen = dst[frequency[dst] >= lo]
            if hi is not None:
                chosen = chosen[frequency[chosen] <= hi]
            labels[chosen] = name
    return labels, frequency


def v24_preprocess(training, others, cats):
    """Every learned transformation fits training only, including inner ES."""
    forbidden = [c for c in training if c == "TARGET" or "SK_ID_CURR" in c]
    assert not forbidden, forbidden
    nums = [c for c in training if c not in cats and training[c].nunique(dropna=True) > 1]
    continuous = [c for c in nums if training[c].nunique(dropna=True) > 2]
    binary = [c for c in nums if c not in continuous]
    missing = [c for c in nums if training[c].isna().any()]
    qt = QuantileTransformer(n_quantiles=min(256, len(training)), output_distribution="normal",
                             subsample=10**9, random_state=42)
    qt.fit(training[continuous].replace([np.inf, -np.inf], np.nan).to_numpy(dtype=float))
    enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32)
    enc.fit(training[cats].to_numpy())

    def transform(frame):
        z = qt.transform(frame[continuous].replace([np.inf, -np.inf], np.nan).to_numpy(dtype=float))
        parts = [np.nan_to_num(z, nan=0.0, posinf=4.0, neginf=-4.0).clip(-4, 4).astype(np.float32)]
        if binary:
            parts.append(frame[binary].fillna(0).to_numpy(dtype=np.float32))
        if missing:
            parts.append(frame[missing].isna().to_numpy(dtype=np.float32))
        parts.append(enc.transform(frame[cats].to_numpy()))
        result = np.ascontiguousarray(np.column_stack(parts), dtype=np.float32)
        assert np.isfinite(result).all()
        return torch.from_numpy(result)

    metadata = dict(numeric=nums, continuous=continuous, binary=binary, missing=missing,
                    category_sizes=[len(c) for c in enc.categories_], fit_rows=len(training))
    return transform(training), [transform(x) for x in others], metadata


class V24FactorizationMachine(nn.Module):
    def __init__(self, dimensions, rank, prevalence):
        super().__init__()
        self.linear = nn.Linear(dimensions, 1)
        self.factors = nn.Parameter(torch.empty(dimensions, rank))
        nn.init.normal_(self.factors, std=0.01)
        nn.init.zeros_(self.linear.weight)
        nn.init.constant_(self.linear.bias, float(np.log(prevalence/(1-prevalence))))

    def forward(self, x):
        interaction = 0.5*((x @ self.factors).square() - x.square() @ self.factors.square()).sum(dim=1)
        return self.linear(x).squeeze(-1) + interaction


def v24_predict(model, data):
    model.eval()
    with torch.no_grad():
        # Logits have the same ranking as sigmoid and avoid numerical ties.
        return torch.cat([model(x) for x in data.split(4096)]).numpy()


def v24_train(data, target, valid=None, y_valid=None, fixed_epochs=None):
    torch.set_num_threads(2)
    torch.use_deterministic_algorithms(True)
    torch.manual_seed(42)
    np.random.seed(42)
    cfg = V24_PROTOCOL["fm"]
    model = V24FactorizationMachine(data.shape[1], cfg["rank"], float(np.mean(target)))
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    target = torch.as_tensor(target, dtype=torch.float32)
    generator = torch.Generator().manual_seed(42)
    best, best_epoch, stale, history = -np.inf, 0, 0, []
    for epoch in range(1, (fixed_epochs or cfg["max_epochs"])+1):
        model.train()
        for ids in torch.randperm(len(target), generator=generator).split(cfg["batch_size"]):
            optimizer.zero_grad(set_to_none=True)
            loss = nn.functional.binary_cross_entropy_with_logits(model(data[ids]), target[ids])
            loss.backward()
            optimizer.step()
        if fixed_epochs is None:
            auc = float(roc_auc_score(y_valid, v24_predict(model, valid)))
            history.append(dict(epoch=epoch, auc=auc))
            print(f"FM inner época {epoch}: AUC {auc:.8f}", flush=True)
            if auc > best + cfg["min_delta"]:
                best, best_epoch, stale = auc, epoch, 0
            else:
                stale += 1
            if stale >= cfg["patience"]:
                break
    # No best checkpoint is needed: every outer/full fit is a new model.
    return model, int(fixed_epochs or best_epoch), history


def v24_frames(ctx, src, dst, forbidden_labels):
    X, sup = ctx["X"]["plusx"], ctx["sup"]
    assert not np.intersect1d(src, forbidden_labels).size
    a = sup.add(X.iloc[src], src, src)
    b = sup.add(X.iloc[dst], src, dst)
    altered = sup.y_all.copy()
    altered[forbidden_labels] = 1-altered[forbidden_labels]
    isolated = Supervised(sup.groups, sup.tkeys, altered)
    pd.testing.assert_frame_equal(a, isolated.add(X.iloc[src], src, src))
    pd.testing.assert_frame_equal(b, isolated.add(X.iloc[dst], src, dst))
    return a, b


def v24_fm_fold(ctx, fold, split):
    a, b = split
    started = time.time()
    ia, ib = next(StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
                  .split(np.zeros(len(a)), ctx["y"][a]))
    inner_train, inner_valid = a[ia], a[ib]
    xa, xv = v24_frames(ctx, inner_train, inner_valid, np.r_[inner_valid, b])
    data, (valid,), inner_meta = v24_preprocess(xa, [xv], ctx["cat_cols"])
    model, epochs, history = v24_train(data, ctx["y"][inner_train], valid, ctx["y"][inner_valid])
    del model, data, valid, xa, xv
    gc.collect()
    xa, xb = v24_frames(ctx, a, b, b)
    data, (valid,), outer_meta = v24_preprocess(xa, [xb], ctx["cat_cols"])
    model, _, _ = v24_train(data, ctx["y"][a], fixed_epochs=epochs)
    prediction = v24_predict(model, valid)
    details = dict(fold=fold, epochs=epochs, inner_history=history, inner_preprocessing=inner_meta,
                   outer_preprocessing=outer_meta, dimensions=int(data.shape[1]),
                   target_flip_audit=True, seconds=time.time()-started,
                   epoch_selection_uses_outer_evaluation_labels=False)
    print(f"FM parte {fold}: AUC {roc_auc_score(ctx['y'][b], prediction):.8f}, épocas {epochs}", flush=True)
    del model, data, valid, xa, xb
    gc.collect()
    return prediction, details


def v24_job(ctx, fold, split):
    started = time.time()
    print(f"V24 parte {fold} iniciada", flush=True)
    # Fit FM first; all baseline learners reset their own streams thereafter.
    fm, fm_details = v24_fm_fold(ctx, fold, split)
    reference = v21_job(ctx, fold, split, [0])
    reference.update(fm=fm, fm_details=fm_details, seconds=time.time()-started)
    return reference


def v24_scores(predictions, fm):
    ranks = np.column_stack([to_rank(p)**2 for p in predictions.T])
    return ranks @ V24_WEIGHTS, to_rank(fm)**2


def v24_choose(target, baseline, fm):
    # This function only receives selection labels; exact ties prefer less FM.
    grid = [dict(alpha=alpha, auc=float(roc_auc_score(target, (1-alpha)*baseline+alpha*fm)))
            for alpha in V24_PROTOCOL["alpha_grid"]]
    return max(grid, key=lambda row: row["auc"]), grid


def v24_auc(target, prediction):
    return float(roc_auc_score(target, prediction)) if len(np.unique(target)) == 2 else None


def v24_diagnostics(y, components, fm, baseline, candidate, strata, fold_ids):
    masks = [(f"parte_{f}", fold_ids == f) for f in sorted(np.unique(fold_ids))]
    masks.append(("partes_0_1", fold_ids < 2))
    if (fold_ids >= 2).any():
        masks += [("partes_2_4", fold_ids >= 2), ("total", np.ones(len(y), dtype=bool))]
    table, correlations, strata_table = [], [], []
    names = V24_PROTOCOL["strata"]["names"]+["ausentes_invalidas"]
    for label, mask in masks:
        if not mask.any():
            continue
        old, new = v24_auc(y[mask], baseline[mask]), v24_auc(y[mask], candidate[mask])
        table.append(dict(parte=label, linhas=int(mask.sum()), v21=old, v24=new,
                          delta=None if old is None or new is None else new-old,
                          fm=v24_auc(y[mask], fm[mask])))
        for index, component in enumerate(V24_LABELS):
            rho = float(spearmanr(fm[mask], components[mask, index]).statistic)
            correlations.append(dict(parte=label, componente=component,
                                     spearman=rho if np.isfinite(rho) else None))
        for name in names:
            chosen = mask & (strata == name)
            old, new = v24_auc(y[chosen], baseline[chosen]), v24_auc(y[chosen], candidate[chosen])
            strata_table.append(dict(parte=label, estrato=name, linhas=int(chosen.sum()),
                                     percentual_linhas=100*float(chosen.sum())/int(mask.sum()),
                                     positivos=int(y[chosen].sum()), v21=old, v24=new,
                                     delta=None if old is None or new is None else new-old,
                                     fm=v24_auc(y[chosen], fm[chosen])))
    return dict(table=table, rank_correlations=correlations, strata_table=strata_table)


def v24_run(ctx, train, directory):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    assert not (directory/"experiment_report.json").exists(), "Use a new research directory; preserve prior evidence."
    protocol_sha = v24_frozen_protocol(directory)
    started = time.time()
    y, n = ctx["y"], ctx["n"]
    splits = list(StratifiedKFold(**V24_PROTOCOL["folds"]).split(np.zeros(n), y))
    ids = np.empty(n, dtype=np.int8)
    for f, (_, b) in enumerate(splits):
        ids[b] = f
    strata, frequency = v24_strata(train, splits)
    # Assignment is frozen before the first fit and contains no outcomes.
    assignment = pd.DataFrame(dict(row=np.arange(n), fold=ids, outer_train_frequency=frequency,
                                   stratum=strata))
    assignment_path = directory/"frozen_strata.csv"
    if assignment_path.exists():
        pd.testing.assert_frame_equal(assignment, pd.read_csv(assignment_path), check_dtype=False)
    else:
        assignment.to_csv(assignment_path, index=False)
    selection = ids < 2
    predictions, fm = np.full((n, 7), np.nan), np.full(n, np.nan)
    iterations, fm_epochs = [[] for _ in range(7)], []
    report = dict(status="running_selection", confirmed=False, csv_sha256=None,
                  protocol=V24_PROTOCOL, protocol_sha256=protocol_sha,
                  frozen_strata_sha256=hashlib.sha256((directory/"frozen_strata.csv").read_bytes()).hexdigest(),
                  input_sha256=input_hashes, exact_installed_versions={p:importlib.metadata.version(p) for p in PINNED},
                  python=sys.version, platform=platform.platform(), fold_results=[],
                  confirmation_folds_executed=[], selection_folds=[0, 1],
                  source_note="All seven v21 components and FM refit from input; no saved predictions or models loaded.")

    def save():
        report["seconds"] = time.time()-started
        v24_json(directory/"experiment_report.json", report)

    def receive(results):
        for result in results:
            f = result["fold"]
            b = splits[f][1]
            predictions[b] = np.column_stack([result["baseline"], result["predictions"][0]])
            fm[b] = result["fm"]
            for dest, values in zip(iterations[:6], result["baseline_iterations"]):
                dest.extend(values)
            iterations[6].append(result["candidate_sizes"][0])
            fm_epochs.append(result["fm_details"]["epochs"])
            report["fold_results"].append({k:result[k] for k in ["fold", "seconds", "threads", "threadpools", "torch_interop_pool", "fm_details"]})
            if f >= 2:
                report["confirmation_folds_executed"].append(f)
            save()

    save()
    compact = {k:v for k,v in ctx.items() if k != "raw"}
    with joblib.parallel_config(backend="loky", n_jobs=2, inner_max_num_threads=2, initializer=v20_initialize_worker):
        receive(joblib.Parallel(max_nbytes=None)(joblib.delayed(v24_job)(compact, f, splits[f]) for f in [0, 1]))
        base, alternative = v24_scores(predictions[selection], fm[selection])
        chosen, grid = v24_choose(y[selection], base, alternative)
        alpha = chosen["alpha"]
        report.update(screening_choice=chosen, screening_grid=grid,
                      screening_baseline_auc=float(roc_auc_score(y[selection], base)),
                      screening_diagnostics=v24_diagnostics(y[selection], predictions[selection], fm[selection],
                          base, (1-alpha)*base+alpha*alternative, strata[selection], ids[selection]))
        v24_json(directory/"screening_selection.json", report)
        print("V24 TRIAGEM", json.dumps(dict(choice=chosen, grid=grid)), flush=True)
        if alpha == 0 or chosen["auc"] <= report["screening_baseline_auc"]:
            report.update(status="rejected_in_screening", reason="Best FM weight is zero or no gain on 0–1; no 2–4 fits, full training, or CSV.",
                          evaluated_weights=np.r_[(1-alpha)*V24_WEIGHTS, alpha].tolist())
            report.update(report["screening_diagnostics"])
            np.savez_compressed(directory/"screening_oof_evidence.npz", rows=np.flatnonzero(selection),
                                fold=ids[selection], v21_components=predictions[selection], fm=fm[selection])
            save()
            return report
        report["status"] = "completing_oof"
        save()
        receive(joblib.Parallel(max_nbytes=None)(joblib.delayed(v24_job)(compact, f, splits[f]) for f in [2, 3, 4]))
    base, alternative = v24_scores(predictions, fm)
    chosen, grid = v24_choose(y[selection], base[selection], alternative[selection])
    alpha = chosen["alpha"]
    report.update(status="recipe_frozen_before_confirmation", alpha=alpha, final_selection_grid=grid,
                  weights=np.r_[(1-alpha)*V24_WEIGHTS, alpha].tolist(),
                  baseline_final_sizes=V24_PROTOCOL["baseline_final_sizes"],
                  fm_final_epochs=max(1, round(float(np.mean(fm_epochs)))),
                  baseline_iterations=iterations, fm_inner_selected_epochs=fm_epochs)
    v24_json(directory/"frozen_selection.json", report)
    save()
    report.update(v24_diagnostics(y, predictions, fm, base, (1-alpha)*base+alpha*alternative, strata, ids))
    rows = {row["parte"]:row for row in report["table"]}
    assert abs(rows["partes_2_4"]["v21"]-V24_PROTOCOL["promotion"]["reference_2_4"]) < 1e-12, "v21 not reproduced."
    assert abs(rows["total"]["v21"]-V24_PROTOCOL["promotion"]["reference_total"]) < 1e-12, "v21 not reproduced."
    approved = alpha > 0 and rows["partes_2_4"]["delta"] >= 0.00010 and rows["total"]["delta"] > 0
    report.update(confirmed=bool(approved), status="gain_confirmed" if approved else "rejected_in_confirmation")
    np.savez_compressed(directory/"oof_evidence.npz", fold=ids, v21_components=predictions, fm=fm)
    save()
    print(pd.DataFrame(report["table"]).to_string(index=False), flush=True)
    return report


def v24_fit_full_fm(ctx, epochs):
    a = np.arange(ctx["n"])
    b = np.arange(ctx["n"], ctx["n"]+len(ctx["Xt"]["plusx"]))
    xa = ctx["sup"].add(ctx["X"]["plusx"], a, a)
    xt = ctx["sup"].add(ctx["Xt"]["plusx"], a, b)
    data, (test_data,), _ = v24_preprocess(xa, [xt], ctx["cat_cols"])
    model, _, _ = v24_train(data, ctx["y"], fixed_epochs=int(epochs))
    return v24_predict(model, test_data)


In [22]:

started=time.time()
evidence_dir=BASE_DIR/"evidence"
protocol_sha256=v24_frozen_protocol(evidence_dir)
train=pd.read_csv(BASE_DIR/"input/train.csv")
test=pd.read_csv(BASE_DIR/"input/test.csv")
assert train.shape==(171202,34) and test.shape==(61500,33)
input_hashes={name:hashlib.sha256((BASE_DIR/"input"/name).read_bytes()).hexdigest() for name in ["train.csv","test.csv"]}
assert input_hashes=={"train.csv":"b8dd5a61771d8eb2821972c1b7d3f2f747cc66471f4d34153036d2d9c458ecf1", "test.csv":"dfb7aeb96e5af3d8616acaaba6b23cec8680845cfd525f7c08de15c6d2d42c65"}
# Freeze diagnostic strata before prepare(), which fits auxiliary regressors.
frozen_splits=list(StratifiedKFold(**V24_PROTOCOL["folds"]).split(np.zeros(len(train)),train.TARGET))
frozen_labels,frozen_frequency=v24_strata(train,frozen_splits)
frozen_ids=np.empty(len(train),dtype=np.int8)
for f,(_,b) in enumerate(frozen_splits):frozen_ids[b]=f
pd.DataFrame(dict(row=np.arange(len(train)),fold=frozen_ids,outer_train_frequency=frozen_frequency,stratum=frozen_labels)).to_csv(evidence_dir/"frozen_strata.csv",index=False)
print("Protocolo e estratos congelados antes de prepare/fit",protocol_sha256,flush=True)
print(pd.Series(frozen_labels).value_counts().to_string(),flush=True)
ctx=prepare(train,test)
add_tabm_feature_set(ctx)
report=v24_run(ctx,train,evidence_dir)
report["seconds_including_prepare"]=time.time()-started
execution_report=dict(report,run_cv=True,cv_only=True,csv_sha256=None)
v24_json(evidence_dir/"experiment_report.json",execution_report)


Protocolo e estratos congelados antes de prepare/fit 02f9588325cc88f8d2d57d83b1cb58108a96ac20c04dd0141e08c1646da3fe5e
muito_frequentes_200_mais    99603
comuns_50_199                47841
pouco_frequentes_10_49       19524
raras_0_9                     4234
{'base': 74, 'plusx': 142, 'catx': 134} 48s


KeyboardInterrupt: 

Pesquisa v24: evidências de execução ficam em evidence/. Nenhuma submissão automática.